
# Car Detection Model Training and Evaluation

This notebook is an improved version of the original training notebook. It does three things:

1. Extracts and inspects the dataset  
2. Creates a **correct YOLO dataset YAML** based on the actual folder structure  
3. Trains and evaluates the model, then explains the results

---

## Project Goal
The original model was mainly good at detecting **top-view cars**, but it struggled with **side-view cars**.  
This notebook helps you:
- train the model more correctly
- evaluate it in a reproducible way
- explain the result clearly in your presentation/report



## Why this notebook was updated

In the original notebook, the YAML file was:

```yaml
train: /content/topview_dataset/train/images
val: /content/topview_dataset/train/images
```

This means the model was being validated on the **same data used for training**, which is not ideal.  
That setup can make the validation look better than reality and does not test generalization properly.

Also, if the dataset mostly contains **top-view cars**, the model becomes biased toward that viewpoint and may miss cars from other angles.


In [ ]:

# Step 1: Upload the dataset ZIP file
from google.colab import files
uploaded = files.upload()

print("Uploaded files:", list(uploaded.keys()))


In [ ]:

# Step 2: Extract the ZIP file
import zipfile
import os
from pathlib import Path

zip_name = next(iter(uploaded.keys()))
extract_path = "/content/topview_dataset"

if os.path.exists(extract_path):
    import shutil
    shutil.rmtree(extract_path)

with zipfile.ZipFile(zip_name, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction complete:", extract_path)

# Show a small tree preview
for root, dirs, files_in_root in os.walk(extract_path):
    level = root.replace(extract_path, "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files_in_root[:5]:
        print(f"{indent}  {f}")
    if level >= 2:
        continue



## Dataset structure inspection

The next cell automatically inspects the dataset and tries to detect one of these common YOLO layouts:

### Layout A
```text
dataset/
├── train/images
├── train/labels
├── valid/images
├── valid/labels
├── test/images
└── test/labels
```

### Layout B
```text
dataset/
├── images/train
├── images/valid
├── images/test
├── labels/train
├── labels/valid
└── labels/test
```

### Layout C
```text
dataset/
├── images
└── labels
```

If the dataset is only in `images/` and `labels/`, evaluation can still run, but it will not be a strong scientific split unless you create separate train/val/test folders.


In [ ]:

# Step 3: Detect dataset structure and create a correct YAML automatically
import os
from pathlib import Path
import yaml

base = Path("/content/topview_dataset")

def exists(p):
    return p.exists()

yaml_dict = {
    "nc": 1,
    "names": ["car"]
}

detected_layout = None

# Layout A: train/images, valid/images, test/images
if exists(base / "train" / "images") and exists(base / "valid" / "images"):
    detected_layout = "A"
    yaml_dict["path"] = str(base)
    yaml_dict["train"] = "train/images"
    yaml_dict["val"] = "valid/images"
    if exists(base / "test" / "images"):
        yaml_dict["test"] = "test/images"

# Layout B: images/train, images/valid, images/test
elif exists(base / "images" / "train") and (exists(base / "images" / "valid") or exists(base / "images" / "val")):
    detected_layout = "B"
    yaml_dict["path"] = str(base)
    yaml_dict["train"] = "images/train"
    yaml_dict["val"] = "images/valid" if exists(base / "images" / "valid") else "images/val"
    if exists(base / "images" / "test"):
        yaml_dict["test"] = "images/test"

# Layout C: images + labels directly
elif exists(base / "images") and exists(base / "labels"):
    detected_layout = "C"
    yaml_dict["path"] = str(base)
    yaml_dict["train"] = "images"
    yaml_dict["val"] = "images"
    yaml_dict["test"] = "images"

else:
    raise FileNotFoundError("Could not detect a supported YOLO dataset structure inside /content/topview_dataset")

yaml_path = base / "data_fixed.yaml"
with open(yaml_path, "w") as f:
    yaml.safe_dump(yaml_dict, f, sort_keys=False)

print("Detected layout:", detected_layout)
print("\nGenerated YAML:\n")
print(open(yaml_path).read())



## Install Ultralytics


In [ ]:
!pip install -q ultralytics pyyaml


## Train the model

This uses `yolo26n.pt` as the starting pretrained model.

Why use a pretrained model?
- It already knows **generic visual features**
- It has seen the **car** class before in large datasets like COCO
- But after fine-tuning, performance still depends heavily on **your dataset distribution**

So if your dataset is mostly top-view cars, the model can become biased toward top-view cars even though the base model was pretrained on the class `car`.


In [ ]:

from ultralytics import YOLO

model = YOLO("yolo26n.pt")

train_results = model.train(
    data=str(yaml_path),
    epochs=30,
    imgsz=1024,
    batch=8,
    project="/content/runs",
    name="top_view_car_detection"
)



## Evaluate the trained model

This cell loads `best.pt` and evaluates it on:
- `test` split if available
- otherwise `val`

It then prints:
- Precision
- Recall
- mAP@50
- mAP@50:95


In [ ]:

from ultralytics import YOLO
import json

best_model_path = "/content/runs/top_view_car_detection/weights/best.pt"
best_model = YOLO(best_model_path)

split_to_use = "test" if "test" in yaml_dict else "val"
print("Using split:", split_to_use)

metrics = best_model.val(
    data=str(yaml_path),
    split=split_to_use,
    imgsz=1024,
    conf=0.25,
    iou=0.6,
    plots=True
)

results = {
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "mAP50": float(metrics.box.map50),
    "mAP50_95": float(metrics.box.map),
}

print("\n=== Evaluation Results ===")
print(json.dumps(results, indent=4))

with open("/content/evaluation_results.json", "w") as f:
    json.dump(results, f, indent=4)

print("\nSaved to /content/evaluation_results.json")



## Example inference on one image


In [ ]:

from google.colab import files
uploaded_image = files.upload()
img_path = next(iter(uploaded_image.keys()))
print("Selected image:", img_path)


In [ ]:

import cv2
from google.colab.patches import cv2_imshow

pred_results = best_model(img_path, imgsz=1024, conf=0.10)

for r in pred_results:
    cv2_imshow(r.plot())



# Detailed Markdown Explanation for the Evaluation

## Observed Result from the old model
The old model produced the following evaluation metrics:

- **Precision:** 0.7682
- **Recall:** 0.6284
- **mAP@50:** 0.6877
- **mAP@50:95:** 0.5588

## What these numbers mean

### Precision = 76.8%
This means that when the model predicts a car, it is correct a good portion of the time.  
So the model is not randomly detecting many false cars.

### Recall = 62.8%
This is the more important issue in this project.  
A recall of 62.8% means the model is **missing a noticeable number of cars**.  
In other words, some cars exist in the image, but the model does not detect them.

### mAP@50 = 68.8%
This indicates the overall detection performance is **moderate**.  
The model is usable, but there is clear room for improvement.

### mAP@50:95 = 55.9%
This metric is stricter and gives a more realistic view of localization quality across different IoU thresholds.  
A value around 55.9% suggests the model is not consistently strong across all detection conditions.

---

## Why did this happen?

### 1. The dataset distribution was biased
The main reason is that the training data was dominated by **top-view vehicles**.  
So the model learned a narrow visual pattern:

> car = what a car looks like from the top

When the car appears from the side, the visual features change:
- wheels become visible
- windows and side body become more important
- the shape is different from top-view

Because the model did not see enough of those examples during training, it struggled to detect side-view cars.

### 2. Pretrained does not mean perfect for your domain
The model started from a pretrained YOLO checkpoint, and yes, YOLO has seen the class **car** before.  
However, pretrained knowledge is only a starting point.

The fine-tuning stage adapts the model to **your own dataset**.  
If your dataset is mostly top-view cars, the model becomes biased toward your dataset distribution.

So even though YOLO is pretrained on `car`, it can still perform poorly on side-view cars if your custom training data does not represent them well.

### 3. Limited generalization
The old model did not fully learn the concept of "car under different viewpoints".  
Instead, it learned a more limited representation of the car appearance in one dominant viewpoint.

That means the model had weaker **generalization ability**.

### 4. Validation setup in the original notebook was weak
The original notebook used:

```yaml
train: /content/topview_dataset/train/images
val: /content/topview_dataset/train/images
```

So the model was being validated on training data.  
That is not a good evaluation strategy because:
- it does not properly measure generalization
- it can make the model appear better than it really is

A proper split should use:
- train for learning
- val for tuning
- test for final evaluation

---

## Why was a new model needed?

A new model was needed because the old one:
- missed some vehicles
- struggled with side-view cars
- had weak generalization to different viewpoints

The solution was **not only changing the architecture**, but improving the data itself:
- more diverse viewpoints
- more side-view examples
- better representation of real-world conditions

This helps improve:
- recall
- generalization
- overall detection robustness

---

## Short presentation-ready summary

The old model achieved acceptable precision, but lower recall, which means it missed some vehicles.  
The main reason was that the training data was biased toward top-view cars, so the model did not generalize well to side-view vehicles.  
Although YOLO was pretrained on the class `car`, fine-tuning on a limited viewpoint dataset made the model specialize too much in top-view patterns.  
That is why a new model trained on more diverse data was necessary.
